# **CSC 380 HW\#3 (Spring 2026) Extra Credit**
## Global Earthquake Analysis Dashboard -- Starter code

**Name:**  
**Assignment:** 380 Extra Credit  
**Collaborators (including AI tools used):**  


### Overview

In this assignment you will:
1. Fetch **real-time earthquake data** from the USGS API (no key needed)
2. Parse and clean the data into a **Pandas DataFrame**
3. Classify earthquakes by **magnitude category**
4. Compute **seismic energy release** using the Gutenberg–Richter Law
5. Build a **daily time-series** with rolling statistics
6. Create **five different visualizations** using Matplotlib
7. Plot all earthquakes on a **world map**
8. Write a short **analysis** of your findings

>### **Code style REQUIREMENTS**
> Follow PEP 8 guidelines -- All lines must be **≤ 79 characters**.  
> Use **inline comments** to explain your logic.  
> All charts must use **Matplotlib** (no other plotting libraries).

---
## Step 0 — Imports & Configuration

In [ ]:
# ── Standard library ────────────────────────────────────────
import json
from datetime import datetime, timedelta

# ── Third-party ─────────────────────────────────────────────
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

print("All imports successful.")


---
## Step 1 — Fetch Earthquake Data from the USGS API

The USGS Earthquake Hazards Program provides a **free REST API**
with no API key required.

- API docs: https://earthquake.usgs.gov/fdsnws/event/1/
- Response format: **GeoJSON**
- We want all earthquakes **≥ M4.0** in the **last 30 days**

**TODO:**
1. Use `datetime` and `timedelta` to compute `START_DATE` and
   `END_DATE` dynamically (so the notebook always pulls fresh
   data when re-run). Format them as `"YYYY-MM-DD"` strings.
2. Call `requests.get()` with the params dict provided.
3. Check the HTTP status code — raise an error if it failed.
4. Parse the JSON response and extract the `"features"` list.
5. Print how many earthquakes were returned.


In [ ]:
# TODO (1): compute END_DATE (today) and START_DATE (30 days ago)
# as "YYYY-MM-DD" strings using datetime / timedelta
END_DATE   = ...
START_DATE = ...

print(f"Query window: {START_DATE}  →  {END_DATE}")

BASE_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

params = {
    "format":       "geojson",
    "starttime":    START_DATE,
    "endtime":      END_DATE,
    "minmagnitude": 4.0,
    "orderby":      "time",
}

# TODO (2): make the GET request, check status, parse JSON
response  = ...
data      = ...
features  = ...

print(f"Earthquakes returned: {len(features):,}")


---
## Step 2 — Parse GeoJSON into a Pandas DataFrame

Each GeoJSON *feature* has this structure:

```
feature["properties"]              → mag, place, time, ...
feature["geometry"]["coordinates"] → [longitude, latitude, depth_km]
```

`time` is a **Unix millisecond timestamp** — convert it with:
```python
pd.to_datetime(df["time"], unit="ms", utc=True)
```

**TODO:**
1. Loop over `features` and build a list of dicts (one per quake)
   with columns: `time`, `magnitude`, `place`,
   `longitude`, `latitude`, `depth_km`.
2. Create a DataFrame from the list.
3. Convert the `time` column to datetime (UTC).
4. Add a `date` column containing the **date only** (no time),
   e.g. `df["time"].dt.date` — you'll use this for daily grouping.
5. Drop rows where `magnitude` is missing (`dropna`).
6. Print the shape and first few rows.


In [ ]:
records = []

# TODO (3): loop over features and append a dict for each quake
for feature in features:
    props  = feature["properties"]
    coords = feature["geometry"]["coordinates"]
    records.append({
        # fill in the fields here
    })

df = pd.DataFrame(records)

# TODO (4): convert 'time' column to datetime (UTC)

# TODO (5): add 'date' column (date only, no time component)

# TODO (6): drop rows with missing magnitude

print(f"DataFrame shape: {df.shape}")
print(df.head())


---
## Step 3 — Magnitude Categories

Seismologists classify earthquakes like this:

| Magnitude range | Category |
|-----------------|----------|
| 4.0 – 4.9       | Minor    |
| 5.0 – 5.9       | Moderate |
| 6.0 – 6.9       | Strong   |
| 7.0 – 7.9       | Major    |
| 8.0 +            | Great    |

**TODO:**  
Add a `category` column to `df` using `pd.cut()` with the
bins and labels from the table above. Use `right=False` so
each bin is `[lower, upper)`.  
Then print the count of each category.


In [ ]:
bins   = [4.0, 5.0, 6.0, 7.0, 8.0, 12.0]
labels = ["Minor", "Moderate", "Strong", "Major", "Great"]

# TODO (6): create df["category"] using pd.cut()

print("Category distribution:")
print(df["category"].value_counts().sort_index())


---
## Step 4 — Seismic Energy Release (Gutenberg–Richter Law)

The energy (in Joules) released by a quake of magnitude *M* is:

$$E = 10^{\,(1.5 \times M\; +\; 4.8)}$$

This formula means:
- A **M6** releases ~**31×** more energy than M5
- A **M7** releases ~**1,000×** more energy than M5  
- A **M8** releases ~**31,000×** more energy than M5

The scale is *logarithmic* — small differences in magnitude
hide enormous differences in real-world destructive power.

**TODO:**
1. Add an `energy_J` column to `df` using `np.power()`.  
   *(Hint: `np.power(10, 1.5 * df["magnitude"] + 4.8)`)*
2. Using **NumPy functions only** (`np.min`, `np.max`,
   `np.mean`, `np.sum`), print the min, max, mean, and
   total energy over the 30-day window.
3. Compute and print the M6/M5 and M7/M5 energy ratios to
   confirm the non-linear scaling described above.


In [ ]:
# TODO (6): add energy_J column using np.power()
df["energy_J"] = ...

mags   = df["magnitude"].to_numpy()
energy = df["energy_J"].to_numpy()

# TODO (7): print min, max, mean, total using numpy functions
print("=== Magnitude statistics ===")
# your code here ...

print()
print("=== Energy statistics (Joules) ===")
# your code here ...

# TODO (8): compute and print M6/M5 and M7/M5 energy ratios
print()
print("Energy scaling comparison:")
# your code here ...


---
## Step 5 — Daily Time-Series Aggregation

**TODO:**  
Group `df` by the `date` column and compute these three
aggregations in a single `.agg()` call:

| New column name | Source column | Aggregation |
|-----------------|---------------|-------------|
| `daily_count`   | `magnitude`   | count       |
| `daily_energy`  | `energy_J`    | sum         |
| `daily_max_mag` | `magnitude`   | max         |

Store the result in a new DataFrame called `df_daily`.  
Print the first 10 rows.

> **Hint:** There may be days with zero earthquakes in our
> filtered dataset. You can use `.reindex()` on the full
> date range and `fill_value=0` to fill those gaps.


In [ ]:
# TODO (9): groupby date, aggregate count / sum / max
df_daily = (
    df.groupby("date")
    .agg(
        # fill in your aggregations here
    )
    .reset_index()
)

# Optional (but good practice): fill in days with 0 quakes
all_dates = pd.date_range(
    START_DATE, END_DATE, freq="D"
).date
df_daily = (
    df_daily
    .set_index("date")
    .reindex(all_dates, fill_value=0)
    .reset_index()
    .rename(columns={"index": "date"})
)

print(f"Daily table shape: {df_daily.shape}")
print(df_daily.head(10).to_string(index=False))


---
## Step 6 — 7-Day Rolling Statistics

A **rolling window** averages each day's value with the
preceding 6 days. This smooths out random spikes and makes
underlying trends easier to see.

**TODO:**
1. Add `roll7_count` — 7-day rolling mean of `daily_count`.
2. Add `roll7_energy` — 7-day rolling mean of `daily_energy`.
3. Add `cum_energy` — **cumulative sum** of `daily_energy`
   (you'll use this in chart 7b).

> **Hint:** `DataFrame["col"].rolling(window=7).mean()`  
> Use `min_periods=1` so the first few rows aren't NaN.


In [ ]:
# TODO (10): 7-day rolling mean for count and energy
df_daily["roll7_count"]  = ...
df_daily["roll7_energy"] = ...

# TODO (11): cumulative sum of daily energy
df_daily["cum_energy"] = ...

print(df_daily[
    ["date", "daily_count", "roll7_count",
     "daily_energy", "roll7_energy"]
].tail(10).to_string(index=False))


---
## Step 7 — Visualizations

Create **five charts** using Matplotlib.  All charts must have:
- A descriptive **title**
- Labeled **axes** (with units where applicable)
- A **legend** where multiple series or categories appear

Use this color palette for the five magnitude categories:

```python
cat_colors = {
    "Minor":    "#3498db",   # blue
    "Moderate": "#2ecc71",   # green
    "Strong":   "#f39c12",   # orange
    "Major":    "#e74c3c",   # red
    "Great":    "#8e44ad",   # purple
}
```


### Chart 7a — Daily Count + 7-Day Rolling Mean

In [ ]:
cat_colors = {
    "Minor":    "#3498db",
    "Moderate": "#2ecc71",
    "Strong":   "#f39c12",
    "Major":    "#e74c3c",
    "Great":    "#8e44ad",
}

# TODO (12): bar chart of daily_count (use ax.bar)
#       overlaid with a line for roll7_count (use ax.plot)
#       Label both series and add a legend.
fig, ax = plt.subplots(figsize=(12, 4))

# your code here ...

ax.set_title("Global Earthquake Count per Day (M ≥ 4.0) "
             "– Last 30 Days")
ax.set_xlabel("Date")
ax.set_ylabel("Number of earthquakes")
ax.legend()
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


### Chart 7b — Cumulative Energy Released (log scale)

In [ ]:
# TODO (13): line chart of cum_energy vs date
#       Use ax.set_yscale("log") for the y-axis.
#       Add ax.fill_between() for visual effect (optional).
fig, ax = plt.subplots(figsize=(12, 4))

# your code here ...

ax.set_yscale("log")
ax.set_title("Cumulative Seismic Energy Released – Last 30 Days")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative energy (Joules, log scale)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


### Chart 7c — Magnitude Histogram

In [ ]:
# TODO (14): plot a histogram of earthquake magnitudes.
#       Use bin edges from 4.0 to max magnitude, step 0.5.
#       Color each bar by its magnitude category using
#       cat_colors.  Add a legend.
#
# Hint: loop over cat_colors, filter df by category,
#       and call ax.hist() once per category.

fig, ax = plt.subplots(figsize=(8, 4))

bin_edges = np.arange(4.0, df["magnitude"].max() + 0.5, 0.5)

# your code here ...

ax.set_title("Earthquake Magnitude Distribution – Last 30 Days")
ax.set_xlabel("Magnitude")
ax.set_ylabel("Count")
ax.legend(title="Category")
plt.tight_layout()
plt.show()


### Chart 7d — Daily Category Composition (Stacked Bar)

In [ ]:
# TODO (15): create a stacked bar chart where each bar is one day,
#       and the bar is divided by category (Minor/Moderate/…).
#
# Steps:
#   1. Group df by ["date", "category"] and count rows.
#   2. Use .unstack(fill_value=0) to pivot categories to columns.
#   3. Call .plot(kind="bar", stacked=True, ...) on the result.
#
# Use cat_colors for the bar colors.

fig, ax = plt.subplots(figsize=(12, 4))

# your code here ...

ax.set_title(
    "Daily Earthquake Composition by Category – Last 30 Days"
)
ax.set_xlabel("Date")
ax.set_ylabel("Count")
ax.legend(title="Category")
plt.tight_layout()
plt.show()


### Chart 7e — Depth vs. Magnitude Scatter Plot

In [ ]:
# TODO (16): scatter plot with depth_km on x-axis and magnitude
#       on y-axis.  Color each point by its category using
#       cat_colors.  Use alpha=0.4 so overlapping points
#       are visible.
#
# After the plot, add a markdown cell below answering:
# "What pattern, if any, do you see between depth and
#  magnitude?"

fig, ax = plt.subplots(figsize=(8, 5))

# your code here ...

ax.set_title("Earthquake Depth vs. Magnitude")
ax.set_xlabel("Depth (km)")
ax.set_ylabel("Magnitude")
ax.legend(title="Category")
plt.tight_layout()
plt.show()


**Analysis — Chart 7e:**  
*(Write 2–3 sentences describing the pattern you observe
between earthquake depth and magnitude.)*

Your answer here.


---
## Step 8 — World Map

Plot every earthquake on a world map where:
- **Dot size** is proportional to seismic **energy released**
- **Dot color** represents **magnitude** (use a colormap)

You do **not** need any external mapping library —
a simple `ax.scatter()` on a `(-180,180) × (-90,90)` axes
works fine.  Optionally, fetch and draw a coastline overlay
from the Natural Earth dataset (see hint below).

**Requirements:**
1. Set `ax.set_facecolor(...)` to represent the ocean.
2. Plot earthquakes with `ax.scatter()`, mapping:
   - `x = longitude`, `y = latitude`
   - `s =` size scaled from energy (hint below)
   - `c =` magnitude, with a colormap (e.g. `cm.plasma`)
3. Add a **colorbar** showing the magnitude scale.
4. Add a simple **size legend** showing what dot sizes
   correspond to M4, M5, M6, M7.
5. Include a descriptive title.

> **Size scaling hint:**  
> Energy spans many orders of magnitude, so use a
> square-root scale to keep dot sizes reasonable:
> ```python
> e      = df["energy_J"].to_numpy()
> e_norm = (e - e.min()) / (e.max() - e.min() + 1e-9)
> sizes  = 5 + e_norm ** 0.5 * 295   # range [5, 300]
> ```

> **Optional coastline hint:**  
> ```python
> import requests, json
> url = ("https://raw.githubusercontent.com/nvkelso/"
>        "natural-earth-vector/master/geojson/"
>        "ne_110m_coastline.geojson")
> coast = requests.get(url).json()
> # then loop over coast["features"] and ax.plot() each
> # LineString / MultiLineString segment
> ```


In [ ]:
# TODO (17): build the world map as described above.

fig, ax = plt.subplots(figsize=(18, 9))
ax.set_facecolor("#cde8f5")     # ocean color
ax.set_xlim(-180, 180)
ax.set_ylim(-90,   90)

# Optional: draw coastlines here ...

# TODO (18): compute sizes from energy (see hint above)
e      = df["energy_J"].to_numpy()
sizes  = ...   # scale to a visible range

# TODO (19): scatter plot — x=longitude, y=latitude,
#       s=sizes, c=magnitude, cmap=cm.plasma
sc = ax.scatter(
    # your arguments here
)

# TODO (20): add colorbar
# cbar = plt.colorbar(sc, ax=ax, ...)

# TODO (21): add size legend (ax.scatter([], [], s=...) trick)

ax.axhline(0, color="gray", lw=0.5, ls="--", alpha=0.5)
ax.set_title(
    f"Global Earthquakes – Last 30 Days "
    f"({START_DATE} to {END_DATE})\n"
    "(dot size = seismic energy,  color = magnitude)",
    fontsize=13,
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()


---
## Step 9 — Written Analysis

Answer **all four questions** in this markdown cell.
Write at least 2–3 sentences per question.
Show any calculations in a code cell below where indicated.

---

**Q1.** Where are most earthquakes concentrated geographically?
Does this match what you know about tectonic plates?

*Your answer here.*

---

**Q2.** If a M8.0 earthquake occurred in your dataset,
how much more energy did it release compared to the
**average** M5.0 in your dataset?
Show the calculation in the code cell below.

*Your answer here.*

---

**Q3.** What trend (if any) do you see in the 7-day rolling
earthquake count (Chart 7a)?
Propose one possible explanation for the pattern.

*Your answer here.*

---

**Q4.** Based on Chart 7e, is there a relationship between
earthquake depth and magnitude?
Describe what you observe and whether it surprised you.

*Your answer here.*


In [ ]:
# Q2 calculation — energy comparison
# TODO: compute the energy of a M8.0 quake and the average
#       energy of M5.0 quakes in your dataset, then find
#       the ratio.

# your code here ...
